In [ ]:
import pandas as pd
import json
from datetime import datetime
pd.set_option('display.max_columns', None)

_EXTRACTED_META_COLS = ["_filter_param", "_filter_value", "_extract_datetime"]

def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df

def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = datetime.today()
    df["_load_datetime"] = pd.to_datetime(load_datetime)
    return df


In [ ]:
df_work_raw = catalog.load('raw/openalex/work_dev#parquet')

[03/03/26 13:29:50] INFO     Loading data from raw/openalex/work_dev#parquet                   ]8;id=551095;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=475802;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

# Nodo

In [ ]:
def openalex_load_work_topics(df_work_raw):
    df_work_raw = _add_openalex_extracted_metadata(df_work_raw)

    df_work = df_work_raw.loc[:, ['id', 'topics', '_filter_param', '_filter_value', '_extract_datetime']]
    df_work = df_work.convert_dtypes()
    
    # Proceso topics
    df_work2topics_exploded = df_work.explode('topics')
    df_work2topics_norm = pd.json_normalize(df_work2topics_exploded['topics'])
    df_work2topics_exploded = df_work2topics_exploded.reset_index(drop=True)
    df_work2topics_norm.rename(columns={'id':'topic_id'}, inplace=True)
   
    # Creación de df con work y sus topics
    df_work2topics = pd.concat(
        (df_work2topics_exploded.loc[:, ['id', '_filter_param', '_filter_value', '_extract_datetime']], df_work2topics_norm),
        axis=1,
    )

    df_work2topics = _add_openalex_loaded_metadata(df_work2topics)

    return df_work2topics


In [ ]:
df_work2topics = openalex_load_work_topics(df_work_raw)

In [ ]:
df_work2topics

,id,_filter_param,_filter_value,_extract_datetime,display_name,topic_id,score,domain.display_name,domain.id,field.display_name,field.id,subfield.display_name,subfield.id,_load_datetime
0,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Fire effects on ecosystems,https://openalex.org/T10555,0.9999,Physical Sciences,https://openalex.org/domains/3,Environmental Science,https://openalex.org/fields/23,Global and Planetary Change,https://openalex.org/subfields/2306,2026-03-03
1,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Plant Water Relations and Carbon Dynamics,https://openalex.org/T10266,0.9998,Physical Sciences,https://openalex.org/domains/3,Environmental Science,https://openalex.org/fields/23,Global and Planetary Change,https://openalex.org/subfields/2306,2026-03-03
2,https://openalex.org/W2140131090,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Plant responses to elevated CO2,https://openalex.org/T11760,0.9988,Life Sciences,https://openalex.org/domains/1,Agricultural and Biological Sciences,https://openalex.org/fields/11,Plant Science,https://openalex.org/subfields/1110,2026-03-03
3,https://openalex.org/W2762087180,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Ferroptosis and cancer prognosis,https://openalex.org/T11297,0.9999,Health Sciences,https://openalex.org/domains/4,Medicine,https://openalex.org/fields/27,Pulmonary and Respiratory Medicine,https://openalex.org/subfields/2740,2026-03-03
4,https://openalex.org/W2762087180,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,RNA modifications and cancer,https://openalex.org/T11482,0.9947,Life Sciences,https://openalex.org/domains/1,"Biochemistry, Genetics and Molecular Biology",https://openalex.org/fields/13,Molecular Biology,https://openalex.org/subfields/1312,2026-03-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,https://openalex.org/W3036911563,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Big Data and Business Intelligence,https://openalex.org/T11891,0.9896,Social Sciences,https://openalex.org/domains/2,"Business, Management and Accounting",https://openalex.org/fields/14,Management Information Systems,https://openalex.org/subfields/1404,2026-03-03
587,https://openalex.org/W3036911563,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Privacy-Preserving Technologies in Data,https://openalex.org/T10764,0.9769,Physical Sciences,https://openalex.org/domains/3,Computer Science,https://openalex.org/fields/17,Artificial Intelligence,https://openalex.org/subfields/1702,2026-03-03
588,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Advancements in Transdermal Drug Delivery,https://openalex.org/T10704,0.9991,Life Sciences,https://openalex.org/domains/1,"Pharmacology, Toxicology and Pharmaceutics",https://openalex.org/fields/30,Pharmaceutical Science,https://openalex.org/subfields/3003,2026-03-03
589,https://openalex.org/W3095652295,institutions.ror,https://ror.org/03cqe8w59,2026-03-03,Lipid Membrane Structure and Behavior,https://openalex.org/T10407,0.9946,Life Sciences,https://openalex.org/domains/1,"Biochemistry, Genetics and Molecular Biology",https://openalex.org/fields/13,Molecular Biology,https://openalex.org/subfields/1312,2026-03-03
